In [ ]:
# Cell 1 — Mount Drive + Install + Load Model
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/DocSnap_Project', exist_ok=True)

!pip install transformers torch flask flask-ngrok pyngrok streamlit nltk spacy scikit-learn sentencepiece -q
!python -m spacy download en_core_web_sm -q

import torch
import warnings
warnings.filterwarnings('ignore')

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Load best model — T5-base fine-tuned
from transformers import T5Tokenizer, T5ForConditionalGeneration

print("\nLoading fine-tuned T5-base model...")
tokenizer = T5Tokenizer.from_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_base_finetuned')
model = T5ForConditionalGeneration.from_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_base_finetuned')

# Check if CUDA is available and move model to GPU, otherwise keep on CPU
if torch.cuda.is_available():
    model = model.cuda()
    print("Model moved to GPU.")
else:
    print("CUDA not available. Model will run on CPU.")

model.eval()

print("T5-base loaded successfully")
print("Ready to build API and UI")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 126.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 114.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
GPU Available: True
GPU Name: Tesla T4

Loading fine-tuned T5-base model...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Model moved to GPU.
T5-base loaded successfully
Ready to build API and UI


In [ ]:
# Cell 2 — Flask REST API for DocSnap
# NOTE: This cell writes the API code to a file
# Run tomorrow when GPU is available

api_code = '''
from flask import Flask, request, jsonify
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import time

app = Flask(__name__)

# Load model
print("Loading T5-base model...")
tokenizer = T5Tokenizer.from_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_base_finetuned')
model = T5ForConditionalGeneration.from_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_base_finetuned')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print(f"Model loaded on {device}")

def summarize_text(text, max_length=150, min_length=40):
    """Summarize text using fine-tuned T5-base"""
    input_text = "summarize: " + text
    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        max_length=512,
        truncation=True,
        padding=True
    ).to(device)

    start_time = time.time()
    with torch.no_grad():
        summary_ids = model.generate(
            inputs['input_ids'],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )
    end_time = time.time()

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    processing_time = round(end_time - start_time, 2)

    return summary, processing_time

@app.route('/')
def home():
    return jsonify({
        'message': 'DocSnap API is running',
        'version': '1.0',
        'model': 'T5-base (fine-tuned on CNN/DailyMail)',
        'endpoints': {
            '/summarize': 'POST - Summarize text',
            '/health': 'GET - Check API health'
        }
    })

@app.route('/health')
def health():
    return jsonify({
        'status': 'healthy',
        'model_loaded': True,
        'device': device
    })

@app.route('/summarize', methods=['POST'])
def summarize():
    try:
        data = request.get_json()

        if not data or 'text' not in data:
            return jsonify({'error': 'No text provided'}), 400

        text = data['text']

        if len(text.strip()) < 50:
            return jsonify({'error': 'Text too short — minimum 50 characters'}), 400

        max_length = data.get('max_length', 150)
        min_length = data.get('min_length', 40)

        # Truncate if too long
        words = text.split()
        if len(words) > 800:
            text = ' '.join(words[:800])

        summary, processing_time = summarize_text(text, max_length, min_length)

        return jsonify({
            'status': 'success',
            'original_length': len(text.split()),
            'summary': summary,
            'summary_length': len(summary.split()),
            'processing_time_seconds': processing_time,
            'model': 'T5-base (fine-tuned)'
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
'''

# Save API code to file
with open('/content/drive/MyDrive/DocSnap_Project/docsnap_api.py', 'w') as f:
    f.write(api_code)

print("Flask REST API code written and saved to Drive")
print("File: /content/drive/MyDrive/DocSnap_Project/docsnap_api.py")
print("\nAPI Endpoints:")
print("  GET  /         — API info")
print("  GET  /health   — Health check")
print("  POST /summarize — Summarize text")
print("\nInput format:")
print('  {"text": "your document here", "max_length": 150, "min_length": 40}')
print("\nOutput format:")
print('  {"status": "success", "summary": "...", "processing_time_seconds": 2.1}')

Flask REST API code written and saved to Drive
File: /content/drive/MyDrive/DocSnap_Project/docsnap_api.py

API Endpoints:
  GET  /         — API info
  GET  /health   — Health check
  POST /summarize — Summarize text

Input format:
  {"text": "your document here", "max_length": 150, "min_length": 40}

Output format:
  {"status": "success", "summary": "...", "processing_time_seconds": 2.1}


In [ ]:
# Cell 3 — Streamlit UI for DocSnap
# Writes the Streamlit app code to a file

streamlit_code = '''
import streamlit as st
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import time

# Page config
st.set_page_config(
    page_title="DocSnap — AI Document Summarizer",
    page_icon="📄",
    layout="wide"
)

# Header
st.title("📄 DocSnap")
st.subheader("AI-Powered Document Summarization System")
st.markdown("---")

# Load model (cached so it only loads once)
@st.cache_resource
def load_model():
    tokenizer = T5Tokenizer.from_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_base_finetuned')
    model = T5ForConditionalGeneration.from_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_base_finetuned')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.eval()
    return model, tokenizer, device

model, tokenizer, device = load_model()

# Sidebar
st.sidebar.title("Settings")
max_length = st.sidebar.slider("Maximum summary length (words)", 50, 250, 150)
min_length = st.sidebar.slider("Minimum summary length (words)", 20, 100, 40)
st.sidebar.markdown("---")
st.sidebar.info("Model: T5-base (fine-tuned on CNN/DailyMail)")
st.sidebar.info("ROUGE-1 Score: 0.4584")

# Main layout
col1, col2 = st.columns(2)

with col1:
    st.markdown("### Input Document")
    input_text = st.text_area(
        "Paste your document here:",
        height=300,
        placeholder="Enter your text here (minimum 50 words)..."
    )

    word_count = len(input_text.split()) if input_text else 0
    st.caption(f"Word count: {word_count}")

with col2:
    st.markdown("### Summary Output")

    if st.button("Summarize", type="primary", use_container_width=True):
        if len(input_text.split()) < 10:
            st.error("Please enter at least 10 words.")
        else:
            with st.spinner("Generating summary..."):
                # Truncate if needed
                words = input_text.split()
                if len(words) > 800:
                    input_text = ' '.join(words[:800])

                input_enc = "summarize: " + input_text
                inputs = tokenizer(
                    input_enc,
                    return_tensors="pt",
                    max_length=512,
                    truncation=True
                ).to(device)

                start = time.time()
                with torch.no_grad():
                    output = model.generate(
                        inputs["input_ids"],
                        max_length=max_length,
                        min_length=min_length,
                        length_penalty=2.0,
                        num_beams=4,
                        early_stopping=True
                    )
                end = time.time()

                summary = tokenizer.decode(output[0], skip_special_tokens=True)
                processing_time = round(end - start, 2)

            # Display results
            st.text_area("Generated Summary:", value=summary, height=200)

            # Metrics
            st.markdown("---")
            m1, m2, m3 = st.columns(3)
            m1.metric("Input Words", len(input_text.split()))
            m2.metric("Summary Words", len(summary.split()))
            m3.metric("Processing Time", f"{processing_time}s")

            compression = round(len(input_text.split()) / len(summary.split()), 1)
            st.success(f"Compression ratio: {compression}x — summarized in {processing_time} seconds")

# Footer
st.markdown("---")
st.markdown("**DocSnap** | MBA Data Science Major Project | Amity University Online 2026 | Model: T5-base (ROUGE-1: 0.4584)")
'''

# Save to Drive
with open('/content/drive/MyDrive/DocSnap_Project/docsnap_app.py', 'w') as f:
    f.write(streamlit_code)

print("Streamlit UI code written and saved to Drive")
print("File: /content/drive/MyDrive/DocSnap_Project/docsnap_app.py")
print("\nUI Features:")
print("  - Two column layout (input left, output right)")
print("  - Adjustable summary length via sidebar")
print("  - Word count display")
print("  - Processing time metric")
print("  - Compression ratio display")
print("  - Model info in sidebar")

Streamlit UI code written and saved to Drive
File: /content/drive/MyDrive/DocSnap_Project/docsnap_app.py

UI Features:
  - Two column layout (input left, output right)
  - Adjustable summary length via sidebar
  - Word count display
  - Processing time metric
  - Compression ratio display
  - Model info in sidebar


In [ ]:
# Set ngrok authtoken
from pyngrok import ngrok
ngrok.set_auth_token("3C2OUMQDgOtBL8iSr2F1sS9GwTh_ZjxsT8mGhz4cPjG8izdR")
print("Ngrok token set successfully")

Ngrok token set successfully


In [ ]:
# Cell 4 — Run Flask REST API with ngrok
import subprocess
import threading
import time
from pyngrok import ngrok

print("=== STARTING FLASK REST API ===\n")

# Copy API file to local
import shutil
shutil.copy(
    '/content/drive/MyDrive/DocSnap_Project/docsnap_api.py',
    '/content/docsnap_api.py'
)

# Start Flask in background
def run_flask():
    subprocess.run(['python', '/content/docsnap_api.py'])

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

# Wait for Flask to start
print("Starting Flask server...")
time.sleep(5)

# Create ngrok tunnel
public_url = ngrok.connect(5000)
print(f"\nFlask REST API is LIVE")
print(f"Public URL: {public_url}")
print(f"\nTest endpoints:")
print(f"  {public_url}/         — API info")
print(f"  {public_url}/health   — Health check")
print(f"\nTo summarize — send POST request to:")
print(f"  {public_url}/summarize")
print(f'\n  Body: {{"text": "your document text here"}}')

=== STARTING FLASK REST API ===

Starting Flask server...

Flask REST API is LIVE
Public URL: NgrokTunnel: "https://prepolitical-kiley-unrustically.ngrok-free.dev" -> "http://localhost:5000"

Test endpoints:
  NgrokTunnel: "https://prepolitical-kiley-unrustically.ngrok-free.dev" -> "http://localhost:5000"/         — API info
  NgrokTunnel: "https://prepolitical-kiley-unrustically.ngrok-free.dev" -> "http://localhost:5000"/health   — Health check

To summarize — send POST request to:
  NgrokTunnel: "https://prepolitical-kiley-unrustically.ngrok-free.dev" -> "http://localhost:5000"/summarize

  Body: {"text": "your document text here"}


In [ ]:
# Cell 5 — Test the REST API
import requests
import json

print("=== TESTING FLASK REST API ===\n")

BASE_URL = "https://prepolitical-kiley-unrustically.ngrok-free.dev"

# Test 1 — Health check
print("Test 1: Health Check")
response = requests.get(f"{BASE_URL}/health")
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}\n")

# Test 2 — Home endpoint
print("Test 2: API Info")
response = requests.get(f"{BASE_URL}/")
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}\n")

# Test 3 — Summarize a real document
print("Test 3: Summarize Real Document")
test_text = """
Artificial intelligence is transforming the way businesses operate across
every industry. Machine learning algorithms can now process vast amounts of
data in seconds, identifying patterns that would take human analysts weeks
to discover. In the financial sector, AI systems are being used to detect
fraudulent transactions, assess credit risk, and automate trading strategies.
Healthcare organizations are deploying AI to analyze medical images, predict
patient outcomes, and accelerate drug discovery. The technology is not without
its challenges however. Concerns about data privacy, algorithmic bias, and
job displacement continue to generate debate among policymakers and industry
leaders. Despite these concerns, investment in AI continues to grow rapidly,
with global spending expected to exceed 500 billion dollars by 2027. Companies
that fail to adopt AI technologies risk falling behind more agile competitors
who are already leveraging these tools to improve efficiency and reduce costs.
"""

payload = {"text": test_text}
response = requests.post(
    f"{BASE_URL}/summarize",
    json=payload,
    headers={"Content-Type": "application/json"}
)

print(f"Status: {response.status_code}")
result = response.json()
print(f"Original length: {result['original_length']} words")
print(f"Summary: {result['summary']}")
print(f"Summary length: {result['summary_length']} words")
print(f"Processing time: {result['processing_time_seconds']} seconds")
print(f"\nAPI test complete — REST API working correctly")

=== TESTING FLASK REST API ===

Test 1: Health Check
Status: 200
Response: {'device': 'cuda', 'model_loaded': True, 'status': 'healthy'}

Test 2: API Info
Status: 200
Response: {
  "endpoints": {
    "/health": "GET - Check API health",
    "/summarize": "POST - Summarize text"
  },
  "message": "DocSnap API is running",
  "model": "T5-base (fine-tuned on CNN/DailyMail)",
  "version": "1.0"
}

Test 3: Summarize Real Document
Status: 200
Original length: 141 words
Summary: Artificial intelligence is transforming the way businesses operate across every industry . The technology is not without its challenges however . Concerns about data privacy, algorithmic bias, and job displacement continue to generate debate .
Summary length: 35 words
Processing time: 2.07 seconds

API test complete — REST API working correctly


In [ ]:
# Cell 6 — Run Streamlit UI
import subprocess
import threading
import time
from pyngrok import ngrok

print("=== STARTING STREAMLIT UI ===\n")

# Copy app file to local
import shutil
shutil.copy(
    '/content/drive/MyDrive/DocSnap_Project/docsnap_app.py',
    '/content/docsnap_app.py'
)

# Start Streamlit in background
def run_streamlit():
    subprocess.run([
        'streamlit', 'run', '/content/docsnap_app.py',
        '--server.port', '8501',
        '--server.headless', 'true'
    ])

streamlit_thread = threading.Thread(target=run_streamlit)
streamlit_thread.daemon = True
streamlit_thread.start()

# Wait for Streamlit to start
print("Starting Streamlit server...")
time.sleep(8)

# Create ngrok tunnel on port 8501
ui_url = ngrok.connect(8501)
print(f"Streamlit UI is LIVE")
print(f"Public URL: {ui_url}")
print(f"\nOpen this URL in your browser to use DocSnap")
print(f"The app has:")
print(f"  - Text input area (left)")
print(f"  - Summary output (right)")
print(f"  - Adjustable settings (sidebar)")
print(f"  - Processing time and compression metrics")

=== STARTING STREAMLIT UI ===

Starting Streamlit server...
Streamlit UI is LIVE
Public URL: NgrokTunnel: "https://prepolitical-kiley-unrustically.ngrok-free.dev" -> "http://localhost:8501"

Open this URL in your browser to use DocSnap
The app has:
  - Text input area (left)
  - Summary output (right)
  - Adjustable settings (sidebar)
  - Processing time and compression metrics


In [ ]:
# Cell 7 — Save URLs and Test Results
print("=== DOCSNAP DEPLOYMENT SUMMARY ===\n")

print("Flask REST API:")
print("  URL: https://prepolitical-kiley-unrustically.ngrok-free.dev")
print("  Endpoints: /, /health, /summarize")
print("  Status: LIVE")

print("\nStreamlit UI:")
print("  URL: https://prepolitical-kiley-unrustically.ngrok-free.dev")
print("  Status: LIVE")

print("\nReal Performance Metrics:")
print("  Processing time: 1.79 seconds (tested live)")
print("  Compression ratio: 3.9x")
print("  Model: T5-base fine-tuned")
print("  ROUGE-1: 0.4584")

# Save deployment info
deployment_info = """
DocSnap Deployment Info
=======================
Flask REST API: https://prepolitical-kiley-unrustically.ngrok-free.dev
Streamlit UI: https://prepolitical-kiley-unrustically.ngrok-free.dev

API Endpoints:
- GET  /         — API info
- GET  /health   — Health check
- POST /summarize — Summarize text

Real Performance:
- Processing time: 1.79 seconds
- Compression ratio: 3.9x
- ROUGE-1: 0.4584
- Model: T5-base fine-tuned on CNN/DailyMail
"""

with open('/content/drive/MyDrive/DocSnap_Project/deployment_info.txt', 'w') as f:
    f.write(deployment_info)

print("\nDeployment info saved to Google Drive")
print("Week 9-10 complete")

=== DOCSNAP DEPLOYMENT SUMMARY ===

Flask REST API:
  URL: https://prepolitical-kiley-unrustically.ngrok-free.dev
  Endpoints: /, /health, /summarize
  Status: LIVE

Streamlit UI:
  URL: https://prepolitical-kiley-unrustically.ngrok-free.dev
  Status: LIVE

Real Performance Metrics:
  Processing time: 1.79 seconds (tested live)
  Compression ratio: 3.9x
  Model: T5-base fine-tuned
  ROUGE-1: 0.4584

Deployment info saved to Google Drive
Week 9-10 complete


In [ ]:
# Cell 8 — Create Gradio App for Hugging Face Spaces
gradio_app = '''
import gradio as gr
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import time

# Load model
print("Loading T5-base model...")
tokenizer = T5Tokenizer.from_pretrained("CNShivani7/DocSnap")
model = T5ForConditionalGeneration.from_pretrained("CNShivani7/DocSnap")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print(f"Model loaded on {device}")

def summarize(text, max_length, min_length):
    if len(text.split()) < 10:
        return "Please enter at least 10 words.", "", "", ""

    words = text.split()
    if len(words) > 800:
        text = " ".join(words[:800])

    input_text = "summarize: " + text
    inputs = tokenizer(
        input_text, return_tensors="pt",
        max_length=512, truncation=True
    ).to(device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(
            inputs["input_ids"],
            max_length=int(max_length),
            min_length=int(min_length),
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )
    end = time.time()

    summary = tokenizer.decode(output[0], skip_special_tokens=True)
    processing_time = f"{round(end - start, 2)} seconds"
    input_words = str(len(text.split()))
    summary_words = str(len(summary.split()))

    return summary, processing_time, input_words, summary_words

# Gradio Interface
with gr.Blocks(title="DocSnap — AI Document Summarizer") as demo:
    gr.Markdown("# DocSnap — AI Document Summarizer")
    gr.Markdown("MBA Data Science Major Project | Amity University Online 2026 | Model: T5-base (ROUGE-1: 0.4584)")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="Input Document",
                placeholder="Paste your document here...",
                lines=10
            )
            with gr.Row():
                max_len = gr.Slider(50, 250, value=150, label="Max Summary Length")
                min_len = gr.Slider(20, 100, value=40, label="Min Summary Length")
            summarize_btn = gr.Button("Summarize", variant="primary")

        with gr.Column():
            output_summary = gr.Textbox(label="Generated Summary", lines=8)
            with gr.Row():
                proc_time = gr.Textbox(label="Processing Time")
                input_words = gr.Textbox(label="Input Words")
                output_words = gr.Textbox(label="Summary Words")

    summarize_btn.click(
        fn=summarize,
        inputs=[input_text, max_len, min_len],
        outputs=[output_summary, proc_time, input_words, output_words]
    )

    gr.Markdown("---")
    gr.Markdown("**Model:** T5-base fine-tuned on CNN/DailyMail | **ROUGE-1:** 0.4584 | **Speed:** ~2 seconds per document")

demo.launch()
'''

# Save app.py
with open('/content/app.py', 'w') as f:
    f.write(gradio_app)

# Save requirements.txt
requirements = """transformers
torch
gradio
sentencepiece
"""

with open('/content/requirements.txt', 'w') as f:
    f.write(requirements)

print("app.py created")
print("requirements.txt created")
print("\nNext: Upload fine-tuned model to Hugging Face Hub")

app.py created
requirements.txt created

Next: Upload fine-tuned model to Hugging Face Hub


In [ ]:
# Cell 9 — Upload model to Hugging Face Hub
from huggingface_hub import HfApi, login

print("=== UPLOADING MODEL TO HUGGING FACE HUB ===\n")

# Login to Hugging Face
# Get your token from: https://huggingface.co/settings/tokens
# Create a NEW token with WRITE access
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
login(token=HF_TOKEN)

print("Logged in successfully")
print("Uploading T5-base fine-tuned model...")

# Upload model
model.push_to_hub("CNShivani7/DocSnap", token=HF_TOKEN)
tokenizer.push_to_hub("CNShivani7/DocSnap", token=HF_TOKEN)

print("\nModel uploaded successfully")
print("Model available at: https://huggingface.co/CNShivani7/DocSnap")

=== UPLOADING MODEL TO HUGGING FACE HUB ===

Logged in successfully
Uploading T5-base fine-tuned model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...qt4sogd/model.safetensors:   0%|          |  553kB /  892MB            

README.md: 0.00B [00:00, ?B/s]


Model uploaded successfully
Model available at: https://huggingface.co/CNShivani7/DocSnap


In [ ]:
# Cell 10 — Upload app files to Hugging Face Space
from huggingface_hub import HfApi

api = HfApi()

print("=== UPLOADING APP TO HUGGING FACE SPACE ===\n")

# Upload app.py
api.upload_file(
    path_or_fileobj='/content/app.py',
    path_in_repo='app.py',
    repo_id='CNShivani7/DocSnap',
    repo_type='space',
    token=HF_TOKEN
)
print("app.py uploaded ✓")

# Upload requirements.txt
api.upload_file(
    path_or_fileobj='/content/requirements.txt',
    path_in_repo='requirements.txt',
    repo_id='CNShivani7/DocSnap',
    repo_type='space',
    token=HF_TOKEN
)
print("requirements.txt uploaded ✓")

print("\nSpace is building...")
print("Your DocSnap app will be live at:")
print("https://huggingface.co/spaces/CNShivani7/DocSnap")
print("\nBuilding takes 3-5 minutes — check the URL after that")

=== UPLOADING APP TO HUGGING FACE SPACE ===

app.py uploaded ✓
requirements.txt uploaded ✓

Space is building...
Your DocSnap app will be live at:
https://huggingface.co/spaces/CNShivani7/DocSnap

Building takes 3-5 minutes — check the URL after that
